# ryGPT — Kaggle T4 training

Fine-tune Qwen2.5-1.5B with QLoRA on your anonymized WhatsApp data.

## Before you run

In the right-sidebar of this notebook, confirm:

1. **Accelerator: GPU T4 ×2** (both GPUs — training uses `accelerate launch --multi_gpu` for real data-parallel speedup, not just one GPU)
2. **Internet: On** (needed to download Qwen weights)
3. **Add Data** → your private `rygpt-data` dataset (contains `train.jsonl`, `val.jsonl`, `name_mapping.json`)
4. **(Only if continuing a multi-day run)** **Add Data** → **Your Notebooks** → this notebook's previously saved output. Section 6b below will auto-detect and resume from that checkpoint.

If any of these are missing, edit the notebook settings first — otherwise the cells below will fail with a clear error message.

## Multi-day training

A single Kaggle session caps out at 12 hours. If one session isn't enough to finish, training checkpoints every `--save-steps` (see Section 7), and it auto-resumes from the latest checkpoint found in `--out-dir`. To carry a checkpoint into tomorrow's session:
1. At the end of today's session, click **Save Version** (top-right) so this session's Output — including `models/lora_adapter/checkpoint-*` — is preserved.
2. Tomorrow, open a fresh session of this same notebook, do step 4 above to attach yesterday's output, then run all cells from the top. Section 6b copies the checkpoint into place before training starts, and `06_train_model.py` picks it up automatically.

## 1. Environment check

In [ ]:
!nvidia-smi | head -20
import torch
print()
print('torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    cap = torch.cuda.get_device_capability(0)
    print('Device:', torch.cuda.get_device_name(0))
    print('Compute capability:', cap)
    print(f'Training precision: {"bf16" if cap[0] >= 8 else "fp16"}')
else:
    raise SystemExit('No GPU detected. Settings → Accelerator → GPU T4 ×1')

## 2. Verify internet is on

Without internet, we can't download the base model.

In [ ]:
import urllib.request
try:
    urllib.request.urlopen('https://github.com', timeout=5)
    print('Internet: OK')
except Exception as e:
    raise SystemExit(
        'Internet appears OFF. Settings → Internet → On. Then re-run this cell.\n'
        f'Error: {e}'
    )

## 3. Clone the repo

Idempotent — safe to re-run if the previous attempt failed.

In [ ]:
import os, shutil
REPO_DIR = '/kaggle/working/ryGPT'
if os.path.exists(REPO_DIR):
    print(f'{REPO_DIR} already exists — removing and re-cloning for a clean state')
    shutil.rmtree(REPO_DIR)
!git clone --depth 1 https://github.com/rihaans/ryGPT.git {REPO_DIR}
%cd {REPO_DIR}
!ls

## 4. Install dependencies

Kaggle images already have `torch`/`transformers`. We add `bitsandbytes` for 4-bit quantization and pin the ryGPT `requirements.txt`.

In [ ]:
!pip install -q -r requirements.txt
!pip install -q bitsandbytes
# Verify
import bitsandbytes as bnb
from peft import LoraConfig
print('bitsandbytes:', bnb.__version__)
print('peft imported OK')

## 5. Wire the Kaggle dataset into the expected paths

Auto-detects any `/kaggle/input/*/train.jsonl` so it works regardless of what you named your dataset slug.

In [ ]:
import glob, os, shutil

REPO_DIR = '/kaggle/working/ryGPT'
os.chdir(REPO_DIR)  # anchor cwd — needed because %cd from earlier cells can be lost after re-clone

# Search RECURSIVELY under /kaggle/input/ — handles both top-level and nested layouts.
candidates = glob.glob('/kaggle/input/**/train.jsonl', recursive=True)

if not candidates:
    print('=' * 60)
    print('Could not find train.jsonl in /kaggle/input/.')
    print('=' * 60)
    if os.path.exists('/kaggle/input'):
        print('Contents of /kaggle/input/:')
        for root, dirs, files in os.walk('/kaggle/input'):
            rel = os.path.relpath(root, '/kaggle/input')
            indent = 0 if rel == '.' else rel.count(os.sep) + 1
            print(f'{"  " * indent}{os.path.basename(root) if indent else "input"}/')
            for f in files[:10]:
                print(f'{"  " * (indent + 1)}{f}')
            if len(files) > 10:
                print(f'{"  " * (indent + 1)}... ({len(files) - 10} more files)')
    else:
        print('/kaggle/input does not exist — no datasets attached to this notebook.')
    print()
    print('Fix:')
    print('  1. Upload train.jsonl + val.jsonl + name_mapping.json to kaggle.com/datasets')
    print('     (create private dataset named `rygpt-data`)')
    print('  2. In this notebook: right sidebar -> Add Data -> search rygpt-data -> Add')
    print('  3. Re-run this cell.')
    raise SystemExit(1)

dataset_dir = os.path.dirname(candidates[0])
print(f'Found dataset at: {dataset_dir}')

# Absolute paths — robust to cwd getting lost.
processed_dir = os.path.join(REPO_DIR, 'data', 'processed')
anon_dir = os.path.join(REPO_DIR, 'data', 'anonymized')
os.makedirs(processed_dir, exist_ok=True)
os.makedirs(anon_dir, exist_ok=True)

for src_name, dst_path in [
    ('train.jsonl', os.path.join(processed_dir, 'train.jsonl')),
    ('val.jsonl', os.path.join(processed_dir, 'val.jsonl')),
    ('name_mapping.json', os.path.join(anon_dir, 'name_mapping.json')),
]:
    matches = glob.glob(f'{dataset_dir}/**/{src_name}', recursive=True)
    if not matches:
        print(f'  MISSING in dataset: {src_name}  (name_mapping.json is optional)')
        continue
    src = matches[0]
    shutil.copy(src, dst_path)
    print(f'  {src_name}  ({os.path.getsize(src)/1e6:.1f} MB)  ->  {dst_path}')

print()
print('cwd is now:', os.getcwd())
!ls -lh {processed_dir} {anon_dir}

## 6. Sanity-check the data

Confirms the training script can read the JSONL correctly before we spend hours training.

In [ ]:
import sys, json
sys.path.insert(0, '/kaggle/working/ryGPT')
from src.dataset import read_jsonl, example_to_chat_messages

train = read_jsonl('data/processed/train.jsonl')
val = read_jsonl('data/processed/val.jsonl')
print(f'train: {len(train):,}  |  val: {len(val):,}')
print()
print('First train example (chat template):')
for m in example_to_chat_messages(train[0]):
    print(f'  [{m["role"]:>9}] {m["content"][:100]}')

## 6b. Resume from a previous session (optional)

If you attached a previous session's output in step 4 above, this cell finds the latest `checkpoint-*` under `/kaggle/input/` and copies it into `models/lora_adapter/` so training resumes instead of starting over. On a fresh run (nothing attached), it prints a message and does nothing — safe to always run.

In [ ]:
import glob, os, shutil

REPO_DIR = '/kaggle/working/ryGPT'
os.chdir(REPO_DIR)

# A previous session's Output, attached as an input, contains
# models/lora_adapter/checkpoint-<step>/ dirs. Find the furthest-along one.
checkpoint_dirs = glob.glob('/kaggle/input/**/lora_adapter/checkpoint-*', recursive=True)

if checkpoint_dirs:
    def step_num(p):
        return int(os.path.basename(p.rstrip('/')).split('checkpoint-')[-1])

    checkpoint_dirs.sort(key=step_num)
    latest = checkpoint_dirs[-1]
    src_adapter_dir = os.path.dirname(latest)  # the lora_adapter/ dir containing it
    dst_adapter_dir = os.path.join(REPO_DIR, 'models', 'lora_adapter')

    if os.path.exists(dst_adapter_dir):
        shutil.rmtree(dst_adapter_dir)
    shutil.copytree(src_adapter_dir, dst_adapter_dir)

    print(f'Restored checkpoint from previous session: {src_adapter_dir}')
    print(f'  latest step: {step_num(latest)}')
    print(f'  copied to:   {dst_adapter_dir}')
    print()
    print('Training will resume from this checkpoint automatically.')
else:
    print('No previous checkpoint found under /kaggle/input/ — starting fresh.')

## 7. Train (Phase 6)

T4×2-tuned hyperparameters:
- `--batch-size 8 --grad-accum 2` per GPU → effective batch 8×2×2 GPUs = 32
- `--max-seq-length 256` (p99 of real data is 170 tokens — clips only 0.16%)
- `--epochs 2`
- `--eval-steps 2000 --save-steps 2000` — evaluation is a full pass over val data and costs real wall time, so it runs less often than every 1000 steps. Checkpoints (for resuming across sessions) are saved at the same cadence.

Launched via `accelerate launch --multi_gpu --num_processes 2` so **both T4s train in parallel** (real data parallelism — each GPU holds a full model copy and processes a different batch), instead of `device_map="auto"`'s single-GPU-at-a-time model splitting.

If this session doesn't finish in 12 hours: stop it, **Save Version**, and continue tomorrow per the "Multi-day training" note at the top — Section 6b + the script's built-in checkpoint resume handle the rest.

**Idle timeout:** Kaggle kills idle sessions after ~20 min. Either keep the tab open, or click **Save Version → Save & Run All** (top-right) to run headless.

In [ ]:
!accelerate launch --multi_gpu --num_processes 2 scripts/06_train_model.py \
    --base-model Qwen/Qwen2.5-1.5B \
    --batch-size 8 \
    --grad-accum 2 \
    --max-seq-length 256 \
    --epochs 2 \
    --eval-steps 2000 \
    --save-steps 2000 \
    --logging-steps 100 \
    --wandb-disabled \
    --out-dir /kaggle/working/ryGPT/models/lora_adapter

## 8. Evaluate (Phase 7)

Produces `eval/perplexity.md`, `eval/samples.md`, `eval/memorization.md`, and (if you added negative-class corpora) `eval/style_classifier.md`.

Expected wall time: **~45 min** on T4.

In [ ]:
!python scripts/07_evaluate.py

## 9. Preview eval results inline

In [ ]:
from pathlib import Path
for name in ('perplexity', 'memorization', 'samples', 'style_classifier'):
    p = Path(f'/kaggle/working/ryGPT/eval/{name}.md')
    if p.exists():
        print(f'{"=" * 20} eval/{name}.md {"=" * 20}')
        # cap at 4000 chars to keep the notebook readable
        text = p.read_text(encoding='utf-8')
        print(text[:4000])
        if len(text) > 4000:
            print(f'\n... ({len(text) - 4000} more chars in the file)')
        print()

## 10. Package adapter for download

Bundles the LoRA adapter + tokenizer + eval markdown into a single archive under `/kaggle/working/` — which Kaggle exposes as **Output** files you can download from the right sidebar.

In [ ]:
!cd /kaggle/working/ryGPT && tar czf /kaggle/working/rygpt_lora_adapter.tar.gz models/lora_adapter eval
!ls -lh /kaggle/working/rygpt_lora_adapter.tar.gz

## 11. Download to your laptop

1. In the right sidebar of this notebook, expand **Output**.
2. Find `rygpt_lora_adapter.tar.gz` and click the download icon.
3. On your laptop:

```powershell
cd C:\Users\rihaa\Development\Projects\ryGPT
tar -xzf rygpt_lora_adapter.tar.gz
# Extracts models/lora_adapter/ and eval/*.md
python scripts/chat.py
```

Inference runs fine on your 4070 laptop — no GPU rental needed past this point.

## Stop the session

When done, hit **Stop** in the top-right to release the GPUs back to your 30-hour weekly quota (T4×2 draws from the quota at roughly double the rate of T4×1 — factor that into how many sessions you have).

If training isn't finished yet: click **Save Version** *before* stopping, so tomorrow's session can attach this one's output and resume (see "Multi-day training" at the top).